---
# **LAB 4 - CUDA memories**
---

# ▶️ CUDA tools...

In [ ]:
!nvidia-smi

In [ ]:
import numpy as np
import numba
from numba import cuda
import warnings
warnings.filterwarnings("ignore")

print(np.__version__)
print(numba.__version__)

cuda.detect()



In [ ]:
# Suppress Numba deprecation and performance warnings
from numba.core.errors import NumbaDeprecationWarning, NumbaPerformanceWarning
import warnings

warnings.simplefilter('ignore', category=NumbaDeprecationWarning)
warnings.simplefilter('ignore', category=NumbaPerformanceWarning)

Utils for compiling and running Numba CUDA code.

In [ ]:
from numba import cuda

def mem_snapshot(print=True):
    free, total = cuda.current_context().get_memory_info()
    used = total - free
    if print:
        # print GPU memory info
        print("\nMemory occupancy:")
        print(f"    GPU total: {total/1024**3:.3f} GB")
        print(f"    GPU free : {free/1024**3:.3f} GB")
        print(f"    GPU used : {used/1024**3:.3f} GB")
    else:
        return used, free, total

# Quick device spec report (Numba)
def device_info(show=True):
    dev = cuda.get_current_device()   # raises if no CUDA device
    _, total = cuda.current_context().get_memory_info() 
    
    if show:
        print("Device object repr:", dev)
        print("Device name:          ", getattr(dev, "name", "<unknown>"))
        print("Compute capability:   ", getattr(dev, "compute_capability", "<unknown>"))

    # Common numeric properties (use getattr to avoid attribute errors)
    props = {
        "  multi_processor_(SM)_count": ["MULTIPROCESSOR_COUNT"],
        "  max_threads_per_block": ["MAX_THREADS_PER_BLOCK"],
        "  max_block_dim_x":       ["MAX_BLOCK_DIM_X"],
        "  max_block_dim_y":       ["MAX_BLOCK_DIM_Y"],
        "  max_block_dim_z":       ["MAX_BLOCK_DIM_Z"],
        "  max_grid_dim_x":        ["MAX_GRID_DIM_X"],
        "  max_grid_dim_y":        ["MAX_GRID_DIM_Y"],
        "  max_grid_dim_z":        ["MAX_GRID_DIM_Z"],
        "  max_shared_memory_per_block (bytes)": ["MAX_SHARED_MEMORY_PER_BLOCK"],
        "  max_shared_memory_per_SM (bytes)": ["MAX_SHARED_MEMORY_PER_MULTIPROCESSOR"],
        "  warp_size":             ["WARP_SIZE"],
        "  compute_capability":    ["COMPUTE_CAPABILITY", "cc"],
    }
    
    feats = {}
    label = "  total_memory (bytes)"
    feats[label] = total
    if show:
        print(f"{label:40}: {total}")
    for label, keys in props.items():
        val = None
        for k in keys:
            val = getattr(dev, k, None)
            feats[k] = val
            if val is not None:
                break
        if show:
            print(f"{label:40}: {val}")
    return feats

_ = device_info()

# ✅ Parallel reduction with shared memory


## ↘️ TODO...

### Block Reduction with Shared Memory (Numba CUDA)

-   Implement a **block-level reduction** kernel in **Numba CUDA** using **shared memory (SMEM)**.

    -   Input: 1D array `array`
    -   Output: 1D array `out` with **one partial sum per block**
    -   Each block loads its elements into shared memory, then reduces them to a single sum.

<br> 🔹 **Learning Objectives**

-   Allocate and use **shared memory** in a CUDA kernel
-   Use `cuda.grid(1)` to compute a **global index**
-   Apply the **standard reduction loop**: $$
    \text{stride} = \frac{\text{blockDim.x}}{2}, \frac{\text{blockDim.x}}{4}, \dots, 1
    $$
-   **Synchronize threads** with `cuda.syncthreads()`
-   **Write** **one result** per block into `out`

<br> 🔹 **Thread tasks...**

-   Each thread:

    -   **Loads** one element into shared memory

    -   **Performs reduction** in shared memory

    -   Thread `tid == 0` **writes the result** for the block

<br> 🔹 **Allocate Shared Memory**

-   Shared array must have **compile-time constant size**:

```{python}
SMEM_SIZE = 1024
smem = cuda.shared.array(SMEM_SIZE, dtype=np.float32)
```

-   This creates one shared buffer per block

- Nest steps:
    - Load Data into Shared Memory
    - Shared Memory Reduction Loop
    - Write One Result per Block

- Template...

```{python}
import numpy as np
from numba import cuda

TPB = 256
SMEM_SIZE = TPB

@cuda.jit
def blockParReduceSMEM(array, out, n):
    tid = cuda.threadIdx.x
    i = cuda.grid(1)

    smem = cuda.shared.array(SMEM_SIZE, dtype=np.float32)

    # TODO: load (with bounds check + padding)
    # TODO: cuda.syncthreads()

    # TODO: reduction loop

    # TODO: write out[blockIdx.x]
```

## ➡️ Solution...

In [ ]:
import numpy as np
from numba import cuda
import time

@cuda.jit
def blockParReduceSMEM(array, out):
    tid = cuda.threadIdx.x
    size = len(array)
    
    # Declare an array in shared memory
    smem = cuda.shared.array(SMEM_SIZE, dtype=np.float32)
    
    # Define the size of the shared memory array
    if tid < size:
        i = cuda.grid(1) # Global index
        smem[tid] = array[i]
        # Ensure writes to shared memory are visible to all threads before reducing
        cuda.syncthreads()
            
        # reduction loop: stride = blockDim.x/2, blockDim.x/4, ..., 1
        stride = cuda.blockDim.x // 2
        while stride > 0:
            if tid < stride:
                smem[tid] += smem[tid + stride]
            cuda.syncthreads()
            stride //= 2

        # After the loop, the zeroth  element contains the sum
        if tid == 0:
            out[cuda.blockIdx.x] = smem[tid]


In [ ]:
SMEM_SIZE = 1024                # smem size per block
N = SMEM_SIZE * 1024 * 1024    # Number of elements 
a = np.random.randn(N).astype(np.float32)   # array: sum = N(N-1)/2

# verify numpy sum time
tic = time.time()
s_cpu = a.sum()
toc = time.time()
print(f"Numpy sum time: {toc - tic:.4f} seconds")

# GPU setup
threads = SMEM_SIZE
blocks = (N + (threads - 1)) // threads
d_a = cuda.to_device(a)
d_out = cuda.device_array((blocks), dtype=a.dtype)

# launch kernels and time
t0 = time.perf_counter()
blockParReduceSMEM[blocks, threads](d_a, d_out)
cuda.synchronize()
t1 = time.perf_counter()
print(f"Kernel blockParReduceSMEM execution time: {t1 - t0:.4f} seconds")
print("speedup over numpy:", (toc - tic) / (t1 - t0))

# Final reduction on CPU
out = d_out.copy_to_host()  # Final reduction in CPU
s_gpu = out.sum()

# Verify correctness
assert s_gpu == s_cpu, f"{s_gpu} != {s_cpu}"  


# ✅ Matrix multiplication with shared memory (smem)


In [ ]:
import numpy as np
from numba import cuda,  float32
import time

@cuda.jit
def matMul(A, B, C):
    """Perform square matrix multiplication of C = A * B.

    Parameters
    ----------
    A : 2D array
        Input matrix A
    B : 2D array
        Input matrix B
    C : 2D array
        Output matrix C
    """

    i, j = cuda.grid(2)
    if i < C.shape[0] and j < C.shape[1]:
        tmp = 0.0
        for k in range(A.shape[1]):
            tmp += A[i, k] * B[k, j]
        C[i, j] = tmp

#  Matrix sizes:
#     A: (N x P) float32
#     B: (P x M) float32
#     C: (N x M) float32

TPB = 16  # Threads per block
N = TPB * 1000  # Number of rows
M = TPB * 1000  # Number of columns
P = TPB * 1000  # Inner dimension

# Initialize matrices
A = np.random.randn((N,P), dtype=np.float32)   # A matrix 
B = 2* np.random.rand((P,M), dtype=np.float32)   # B matrix 
C = np.zeros((N, M), dtype=np.float32)  # Output matrix

# verify numpy sum time
tic = time.time()
C_cpu = A @ B
toc = time.time()
print(f"Numpy sum time: {toc - tic:.4f} seconds")

# GPU setup
threads = (TPB, TPB)
blocks = ((N + (threads[0] - 1)) // threads[0], (M + (threads[1] - 1)) // threads[1])
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((N, M), dtype=A.dtype)

# launch kernels and time
t0 = time.perf_counter()
matMul[blocks, threads](d_A, d_B, d_C)
cuda.synchronize()
t1 = time.perf_counter()
print(f"Kernel blockParReduceSMEM execution time: {t1 - t0:.4f} seconds")
print("speedup over numpy:", (toc - tic) / (t1 - t0))

# Final reduction on CPU
C = d_C.copy_to_host()  # Final reduction in CPU

# Verify correctness
print(C)
print(C_cpu)


## ↘️ TODO...

<br> 🔹 **Problem Definition**

-   Given: $A$ of shape $(N, P)$, $B$ of shape $(P, M)$

-   Compute: 
$$
      C = A \cdot B \quad \text{of shape } (N, M) 
$$

-   Elementwise: 
    $$
      C[y, x] = \sum_{k=0}^{P-1} A[y, k] \cdot B[k, x]
    $$

-   using **TPB×TPB tiles** loaded into shared memory

<br> 🔹 **Learning Objectives**

-   Launch a kernel with a **2D grid** and **2D blocks**
-   Use `cuda.shared.array()` to create **shared-memory tiles**
-   Implement a tiled dot product using **sweep over tiles**
-   Use `cuda.syncthreads()` correctly
-   Validate GPU results against NumPy (`A @ B`)
-   Measure runtime and compute speedup

<br> 🔹 **CUDA Mapping**

- Each thread computes one element of C:
    - Thread global coordinates:
    ```python
    x, y = cuda.grid(2)
    ```
- Thread local coordinates in the block:

```python
tx = cuda.threadIdx.x
ty = cuda.threadIdx.y
```

- So thread `(tx, ty)` in block `(bx, by)` computes `C[y, x]`

<br> 🔹 **Tiling Strategy (High Level)**

- Instead of reading the full row/column from global memory, we:
	1.	Load a tile of A into shared memory sA
	2.	Load a tile of B into shared memory sB
	3.	Multiply-accumulate inside the tile
	4.	Repeat for all tiles along the inner dimension

- This reduces global memory traffic

<br> 🔹 **Exercise Tasks**

- Your tasks:
	1.	Create shared-memory tiles sA and sB
	2.	Compute global coordinates (x, y) using cuda.grid(2)
	3.	Loop over tiles along the inner dimension
	4.	Load tiles from A and B into shared memory
	5.	Synchronize threads
	6.	Compute partial dot product using shared memory
	7.	Synchronize again before loading next tile
	8.	Store final result in C[y, x]



```{python}
import numpy as np
from numba import cuda, float32

TPB = 16

@cuda.jit
def matMul_SMEM(A, B, C):
    sA = cuda.shared.array(shape=(TPB, TPB), dtype=float32)
    sB = cuda.shared.array(shape=(TPB, TPB), dtype=float32)

    x, y = cuda.grid(2)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    # TODO: compute how many tiles along the inner dimension
    # tiles = ...

    tmp = float32(0.0)

    # TODO: loop over tiles
    # for i in range(tiles):

        # TODO: load sA[ty, tx] from A (with bounds)
        # TODO: load sB[ty, tx] from B (with bounds)

        # TODO: synchronize
        # cuda.syncthreads()

        # TODO: compute partial dot product over j
        # for j in range(TPB):
        #     tmp += sA[ty, j] * sB[j, tx]

        # TODO: synchronize before next tile
        # cuda.syncthreads()

    # TODO: store result in C (with bounds)
```



## ➡️ Solution...

In [ ]:
import numpy as np
from numba import cuda,  float32
import time

# Controls threads per block and shared memory usmemAge.
# The computation will be done on blocks of TILExTILE elements
# TILE should not be larger than 32 in this example

@cuda.jit
def matMul_SMEM(A, B, C):
    """
    Perform matrix multiplication of C = A * B using CUDA shared memory for improved performance. 
    Params:
        A : 2D array
            Input matrix A
        B : 2D array
            Input matrix B
        C : 2D array
            Output matrix C
    """
    # Shared memory tiles
    smemA = cuda.shared.array((TILE, TILE), dtype=float32)
    smemB = cuda.shared.array((TILE, TILE), dtype=float32)

    row, col = cuda.grid(2)   # (y, x) global indices
    ty = cuda.threadIdx.y
    tx = cuda.threadIdx.x

    # Dimensions
    M = A.shape[0]
    P = A.shape[1]
    N = B.shape[1]

    if row >= M or col >= N:
        return

    acc = float32(0.0)

    n_tiles = (P + TILE - 1) // TILE
    for t in range(n_tiles):
        kA = t * TILE + tx      # column index in A
        kB = t * TILE + ty      # row index in B

        # Load A tile
        if kA < P:
            smemA[ty, tx] = A[row, kA]
        else:
            smemA[ty, tx] = 0.0

        # Load B tile
        if kB < P:
            smemB[ty, tx] = B[kB, col]
        else:
            smemB[ty, tx] = 0.0

        # Synchronize to make sure the tiles are loaded
        cuda.syncthreads()

        # Multiply the two tiles
        for j in range(TILE):
            acc += smemA[ty, j] * smemB[j, tx]

        # Synchronize before loading the next tile
        cuda.syncthreads()

    C[row, col] = acc

        
#  Matrix sizes:
#     A: (N x P) float32
#     B: (P x M) float32
#     C: (N x M) float32

TILE = 16  # Threads per block
N = TILE *1000  # Number of rows
M = TILE *1000  # Number of columns
P = TILE *2000  # Inner dimension
A = np.ones((N,P), dtype=np.float32)        # A matrix 
B = 2* np.ones((P,M), dtype=np.float32)     # B matrix 
C = np.zeros((N, M), dtype=np.float32)      # Output matrix

# verify numpy sum time
tic = time.time()
C_cpu = A @ B
toc = time.time()
print(f"Numpy sum time: {toc - tic:.4f} seconds")

# GPU setup
threads = (TILE, TILE)
#                  cols                                     rows    
blocks = ((M + (threads[0] - 1)) // threads[0], (N + (threads[1] - 1)) // threads[1])
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((N, M), dtype=A.dtype)

# launch kernels and time
t0 = time.perf_counter()
matMul_SMEM[blocks, threads](d_A, d_B, d_C)
cuda.synchronize()
t1 = time.perf_counter()
print(f"Kernel blockParReduceSMEM execution time: {t1 - t0:.4f} seconds")
print("speedup over numpy:", (toc - tic) / (t1 - t0))

# Final reduction on CPU
C = d_C.copy_to_host()  # Final reduction in CPU

# Verify correctness
print(C)
print(C_cpu)

# Question: How many blocks can run at the same time on one GPU SM?

This matters because:
- more active blocks → better GPU utilization
- too many resources per block → fewer blocks can fit → lower performance


**What limits the number of active blocks?**

Each Streaming Multiprocessor (SM) has limited resources:
1.	Shared memory
2.	Threads
3.	Registers
4.	Hardware limit on blocks per SM

Your kernel uses some of these resources per block, so the GPU can only fit a certain number of blocks at once.


1️⃣ **Shared memory limit**
If:
```
SM has 64 KB shared memory
your block uses 8 KB
```
Then:
```64 / 8 = 8 blocks max (due to shared memory)```


2️⃣ **Thread limit**  
If:
```
SM supports 2048 threads
your block has 256 threads 
```
Then:
```
2048 / 256 = 8 blocks max (due to threads)
```


3️⃣ **Register limit**  
If:
```
SM has 65536 registers
your block uses 8192 registers
``` 
Then:
```
max blocks per SM due to registers = 65536 / 8192 = 8 blocks
```


4️⃣ **Hardware limit**  

Each GPU has a maximum number of blocks that can be active on an SM, e.g.,
```Max blocks per SM = 32```    

<br>

**What the fixed version does better**

✔ Separates clearly:
-	kernel usage (threads per block, shared memory per block)
-	device limits (threads per SM, shared memory per SM)

✔ Computes each limit independently

✔ Takes the minimum (the real bottleneck)

✔ Works on different GPUs (T4, V100, A100…)

<br>
**Example (very concrete)**

Say:
```
TILE = 16
threads/block = 256
shared memory/block = 2 KB
```

GPU:
```
shared memory/SM = 64 KB
threads/SM = 2048
max blocks/SM = 32
```

Liimits:
```
by shared memory: 64 / 2 = 32 blocks
by threads: 2048 / 256 = 8 blocks
by hardware: 32 blocks
```
> ➡ **Result: 8 blocks per SM**

In [ ]:
from numba import cuda

def estimate_active_blocks_per_sm(
    threads_per_block: int,
    smem_per_block_bytes: int,
    regs_per_thread: int | None = None,
    dynamic_smem_bytes: int = 0,
):
    """
    Rough occupancy-style estimate: active blocks per SM limited by:
      - shared memory per SM
      - registers per SM (optional, if regs_per_thread provided + device exposes regs/SM)
      - max threads per SM
      - hardware max blocks per SM

    This is an estimate (not as accurate as CUDA occupancy calculator / Nsight).
    """
    dev = cuda.get_current_device()

    # ---- device limits (best-effort attribute access) ----
    smem_per_sm = getattr(dev, "MAX_SHARED_MEMORY_PER_MULTIPROCESSOR", None)
    if smem_per_sm is None:
        # fallback: many GPUs have at least 64KB/SM, but this is device-dependent
        smem_per_sm = 64 * 1024

    max_blocks_per_sm = getattr(dev, "MAX_BLOCKS_PER_MULTIPROCESSOR", 32)
    max_threads_per_sm = getattr(dev, "MAX_THREADS_PER_MULTIPROCESSOR", 2048)
    sm_count = getattr(dev, "MULTIPROCESSOR_COUNT", 1)

    # registers per SM (may or may not be exposed)
    regs_per_sm = getattr(dev, "MAX_REGISTERS_PER_MULTIPROCESSOR", None)
    max_regs_per_thread = getattr(dev, "MAX_REGISTERS_PER_THREAD", None)

    # ---- constraints ----
    total_smem_per_block = smem_per_block_bytes + dynamic_smem_bytes

    # limit by shared memory
    if total_smem_per_block <= 0:
        limit_smem = max_blocks_per_sm
    else:
        limit_smem = smem_per_sm // total_smem_per_block

    # limit by threads/SM
    if threads_per_block <= 0:
        limit_threads = 0
    else:
        limit_threads = max_threads_per_sm // threads_per_block

    limits = [max_blocks_per_sm, limit_smem, limit_threads]

    # optional: limit by registers
    limit_regs = None
    if regs_per_thread is not None and regs_per_sm is not None:
        if max_regs_per_thread is not None:
            # if user asks for more than allowed, force 0 blocks
            if regs_per_thread > max_regs_per_thread:
                limit_regs = 0
            else:
                regs_per_block = regs_per_thread * threads_per_block
                limit_regs = regs_per_sm // regs_per_block if regs_per_block > 0 else 0
        else:
            regs_per_block = regs_per_thread * threads_per_block
            limit_regs = regs_per_sm // regs_per_block if regs_per_block > 0 else 0

        limits.append(limit_regs)

    active_blocks_per_sm = max(0, int(min(limits)))
    total_active_blocks = active_blocks_per_sm * sm_count

    return {
        "smem_per_sm_bytes": smem_per_sm,
        "regs_per_sm": regs_per_sm,
        "max_threads_per_sm": max_threads_per_sm,
        "max_blocks_per_sm": max_blocks_per_sm,
        "threads_per_block": threads_per_block,
        "smem_per_block_bytes": total_smem_per_block,
        "limit_by_smem": int(limit_smem),
        "limit_by_threads": int(limit_threads),
        "limit_by_regs": (None if limit_regs is None else int(limit_regs)),
        "active_blocks_per_sm_est": active_blocks_per_sm,
        "total_active_blocks_est": total_active_blocks,
        "sm_count": sm_count,
    }


# ----------------------------
# Example: MatMul shared memory tiles (two TILE x TILE float32 tiles)
# smem/block = 2 * TILE*TILE * 4 bytes
# threads/block = TILE*TILE
# ----------------------------
bytes_per_elem = 4

print(f"{'TILE':>6} {'threads/block':>14} {'SMEM/block KB':>14} {'blk/SM (smem)':>14} {'blk/SM (thr)':>12} {'blk/SM est':>11} {'total blks':>12}")

for TILE in [8, 16, 32]:
    threads_per_block = TILE * TILE
    smem_per_block = 2 * (TILE * TILE) * bytes_per_elem  # sA + sB

    est = estimate_active_blocks_per_sm(
        threads_per_block=threads_per_block,
        smem_per_block_bytes=smem_per_block,
        regs_per_thread=None,           # fill if you know it (see notes below)
        dynamic_smem_bytes=0,
    )

    print(
        f"{TILE:5d} "
        f"{threads_per_block:10d} "
        f"{smem_per_block/1024:14.1f} "
        f"{est['limit_by_smem']:14d} "
        f"{est['limit_by_threads']:12d} "
        f"{est['active_blocks_per_sm_est']:11d} "
        f"{est['total_active_blocks_est']:12d}"
    )

# ✅ Convolution with smem

## ↘️ TODO...


- Host code that:

  - allocates input `data`, `mask`, and output arrays
  - computes a reference result with `np.convolve(..., mode="same")`
  - runs the shared-memory kernel
  - runs the basic kernel
  - prints timings and speed-ups
  - checks maximum absolute error between host and device results

You **do not need** to type all code from scratch – focus on **reading & modifying**.

<br> 🔹  **Understand the Parameters**

- Inspect the parameter definitions:

```python
BLOCK_SIZE   = 1024
MASK_RADIUS  = 100
MASK_SIZE    = 2 * MASK_RADIUS + 1
TILE_SIZE    = BLOCK_SIZE + MASK_SIZE - 1  # shared tile length per block

n = 1024 * 1024 * 1024
```


<br> 🔹  **CPU Reference Convolution**

- The host code computes:

```python
h_ref = np.convolve(data, mask, mode='same')
```

<br> 🔹  **Basic Kernel `conv1d_basic`**

Skeleton:

```python
@cuda.jit
def conv1d_basic(result, data, mask):
    i = cuda.grid(1)
    if i >= len(data):
        return

    mask_size = mask.shape[0]
    radius = mask_size // 2

    offset = i - radius
    start = 0 if offset >= 0 else -offset
    end = mask_size if (offset + mask_size) <= n else (n - offset)

    acc = 0.0
    for j in range(start, end):
        acc += data[offset + j] * mask[j]

    result[i] = acc
```

<br> 🔹  **Shared-Memory Kernel `conv1d_shared`**

- Each block loads a **tile** into shared memory:
  - left halo (MASK_RADIUS elements)
  - center (block size elements)
  - right halo (MASK_RADIUS elements)
- Threads then read from shared memory instead of global memory

### Tasks

1. Identify where **left halo**, **center**, and **right halo** are loaded in `conv1d_shared`.
2. Explain why the code calls `cuda.syncthreads()` before performing the the convolution.
3. Compare memory access patterns:
   - basic kernel: global memory
   - shared kernel: global → shared → reused

<br> 🔹  **Launch Configuration**

- The device launch configuration in the host code is:

```python
threads = BLOCK_SIZE
blocks = (n + BLOCK_SIZE - 1) // BLOCK_SIZE
conv1d_shared[blocks, threads](d_result, d_data, d_mask)
```


<br> 🔹 **Timing & Speedup**

At the end, the script prints:

- `t_host` (CPU time)
- `t_dev_shared` (GPU shared-memory mode)
- `t_dev_basic` (GPU basic kernel)
- Speedups: `host / shared`, `basic / shared`
- Maximum absolute errors vs reference

<br> 🔹  **Experiment: Different Masks**

Currently, `mask` is:

```python
mask = np.ones(MASK_SIZE, dtype=np.float32)
```


## ➡️ Solution...

In [ ]:
from numba import cuda, float32
import numpy as np
import time

@cuda.jit
def conv1d_shared(result, data, mask):
    """
    Shared-memory tiled 1D convolution.
    result, data: 1D device arrays (float32)
    mask: 1D device array length MASK_SIZE
    n: number of elements in data
    """
    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdx = cuda.blockDim.x

    i = bx * bdx + tx

    # allocate shared tile (static size from top-level constant)
    s_tile = cuda.shared.array(shape=TILE_SIZE, dtype=float32)

    # compute left and right boundaries for the block (relative to original code)
    left = bx * bdx - MASK_RADIUS
    right = (bx + 1) * bdx

    # left halo
    if tx < MASK_RADIUS:
        idx = left + tx
        if idx >= 0 and idx < n:
            s_tile[tx] = data[idx]
        else:
            s_tile[tx] = 0.0

    # center
    if i < n:
        s_tile[tx + MASK_RADIUS] = data[i]
    else:
        # threads beyond n should write zero so convolution inside tile is safe
        s_tile[tx + MASK_RADIUS] = 0.0

    # right halo
    if tx >= bdx - MASK_RADIUS:
        # map to tile index for right halo
        idx = right + tx - bdx + MASK_RADIUS
        if idx >= 0 and idx < n:
            s_tile[tx + MASK_SIZE - 1] = data[idx]
        else:
            s_tile[tx + MASK_SIZE - 1] = 0.0

    # wait for all writes to shared memory
    cuda.syncthreads()

    # compute convolution for element i if in-bounds
    if i < n:
        acc = 0.0
        # loop over mask radius
        for m in range(-MASK_RADIUS, MASK_RADIUS + 1):
            acc += s_tile[tx + MASK_RADIUS + m] * mask[m + MASK_RADIUS]
        result[i] = acc

@cuda.jit
def conv1d_basic(result, data, mask):
    """
    Basic 1D convolution kernel (each thread computes one output element).
    Args:
      result: 1D device array (output), length n
      data:   1D device array (input), length n
      mask:   1D device array (mask), length = 2*radius + 1
    Notes:
      - mask.shape[0] is used to derive mask_size and radius on-device.
      - Bounds checks are reduced by computing the valid mask index range.
    """
    i = cuda.grid(1)
    
    if i >= len(data):
        return

    # Local copies (faster than repeated attribute lookups)
    mask_size = mask.shape[0]
    radius = mask_size // 2

    # Compute convolution window [start, start + mask_size)
    offset = i - radius

    # Compute mask index range that maps to valid data indices:
    # j iterates over mask indices such that 0 <= start + j < n
    start = 0 if offset >= 0 else -offset
    end = mask_size if (offset + mask_size) <= n else (n - offset)

    acc = 0.0  # accumulator (float)
    # Loop only over valid mask indices (no per-iteration index bound checks)
    for j in range(start, end):
        acc += data[offset + j] * mask[j]

    result[i] = acc


##---------------------------------
# parameters
#---------------------------------
BLOCK_SIZE = 1024
MASK_RADIUS = 100
MASK_SIZE = 2 * MASK_RADIUS + 1
TILE_SIZE = BLOCK_SIZE + MASK_SIZE - 1  # shared tile length per block

# Array size
n = 1024 * 1024 * 1024  # 1B elements

print(f"Array size = {n}")
print(f"Mask size = {MASK_SIZE}")
print(f"Block size = {BLOCK_SIZE} (threads per block)")
grid_size = (n + BLOCK_SIZE - 1) // BLOCK_SIZE
print(f"Grid: {grid_size} blocks x {BLOCK_SIZE} threads")

##---------------------------------
# allocations and initializations
#---------------------------------
data = np.ones(n, dtype=np.float32)
result = np.ones(n, dtype=np.float32)

# simple moving-average mask (sum=1)
mask = np.full(MASK_SIZE, 1.0 / MASK_SIZE, dtype=np.float32)
t0 = time.perf_counter()
h_ref = np.convolve(data, mask, mode='same')
t_host = time.perf_counter() - t0
print(f"Host convolution time = {t_host:.6f} s")

##---------------------------------
## Device allocations and data transfers
##--------------------------------- 
d_data = cuda.to_device(data)
d_mask = cuda.to_device(mask)
d_result = cuda.device_array(n, dtype=np.float32)
threads = BLOCK_SIZE
blocks = grid_size

##---------------------------------
# shared-memory kernel
#---------------------------------
t0 = time.perf_counter()
conv1d_shared[blocks, threads](d_result, d_data, d_mask)
cuda.synchronize()
t_dev_shared = time.perf_counter() - t0
h_res_shared = d_result.copy_to_host()

##---------------------------------
# basic kernel
#---------------------------------
t0 = time.perf_counter()
conv1d_basic[blocks, threads](d_result, d_data, d_mask)
cuda.synchronize()
t_dev_basic = time.perf_counter() - t0
h_res_basic = d_result.copy_to_host()

# print timings and speedups
print("\nTimings:")
print(f"  CPU (host)          = {t_host:.6f} s")
print(f"  GPU (shared memory) = {t_dev_shared:.6f} s")
print(f"  GPU (basic)         = {t_dev_basic:.6f} s")
if t_dev_shared > 0:
    print(f"  Speed-up (host / shared) = {t_host / t_dev_shared:.6f}")
if t_dev_shared > 0:
    print(f"  Speed-up (basic / shared) = {t_dev_basic / t_dev_shared:.6f}")
# verify correctness
max_err_shared = np.max(np.abs(h_ref - h_res_shared))
max_err_basic = np.max(np.abs(h_ref - h_res_basic))
print(f"\nMax absolute error (shared memory) = {max_err_shared:.6e}")
print(f"Max absolute error (basic)         = {max_err_basic:.6e}")